In [ ]:
import numpy as np
import cv2
import glob
from google.colab.patches import cv2_imshow
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# parametres capteur
largeur_capteur_mm = 6.17
hauteur_capteur_mm = 4.55

# parametres damier
taille_carre_mm = 30.0
colonnes_internes = 8
lignes_internes = 5

criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

objp = np.zeros((lignes_internes * colonnes_internes, 3), np.float32)
objp[:, :2] = np.mgrid[0:colonnes_internes, 0:lignes_internes].T.reshape(-1, 2) * taille_carre_mm

images = glob.glob('/content/drive/MyDrive/seq_data/*')
print(f"{len(images)} images trouvees dans le dossier.")

objpoints = []
imgpoints = []
images_valides = []

In [ ]:
for fname in images:
    img = cv2.imread(fname)
    if img is None:
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ret, corners = cv2.findChessboardCorners(gray, (colonnes_internes, lignes_internes), None)

    if ret == True:
        objpoints.append(objp)
        corners2 = cv2.cornerSubPix(gray, corners, (11,11), (-1,-1), criteria)
        imgpoints.append(corners2)
        images_valides.append(fname)

        img_show = cv2.drawChessboardCorners(img.copy(), (colonnes_internes, lignes_internes), corners2, ret)
        print(f"Succes pour : {fname}")
        cv2_imshow(img_show)
    else:
        print(f"Echec de la detection pour : {fname}")

In [ ]:
# calibration
ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, gray.shape[::-1], None, None)

print("RMS :", ret)
print("Matrice de la camera :\n", mtx)
print("Coefficients de distorsion :\n", dist)

# extraction des caracteristiques du capteur
fovx, fovy, focalLength, principalPoint, aspectRatio = cv2.calibrationMatrixValues(
    cameraMatrix=mtx,
    imageSize=gray.shape[::-1],
    apertureWidth=largeur_capteur_mm,
    apertureHeight=hauteur_capteur_mm
)

focale_metre = focalLength / 1000.0

print(f"Champ de vision X : {fovx:.2f} degres")
print(f"Champ de vision Y : {fovy:.2f} degres")
print(f"Focale de la camera : {focale_metre:.6f} metres")
print(f"Centre optique : {principalPoint}")

In [ ]:
# definition du repere
longueur_axe = 3 * taille_carre_mm
axis = np.float32([[longueur_axe,0,0], [0,longueur_axe,0], [0,0,-longueur_axe]]).reshape(-1,3)

def draw_axes(img, corners, imgpts):
    corner = tuple(corners[0].ravel().astype(int))
    img = cv2.line(img, corner, tuple(imgpts[0].ravel().astype(int)), (255,0,0), 5) # axe X bleu
    img = cv2.line(img, corner, tuple(imgpts[1].ravel().astype(int)), (0,255,0), 5) # axe Y vert
    img = cv2.line(img, corner, tuple(imgpts[2].ravel().astype(int)), (0,0,255), 5) # axe Z rouge
    return img

for fname in images_valides:
    img = cv2.imread(fname)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ret, corners = cv2.findChessboardCorners(gray, (colonnes_internes, lignes_internes), None)

    if ret:
        corners2 = cv2.cornerSubPix(gray, corners, (11,11), (-1,-1), criteria)
        ret_pnp, rvecs_img, tvecs_img = cv2.solvePnP(objp, corners2, mtx, dist)
        imgpts, jac = cv2.projectPoints(axis, rvecs_img, tvecs_img, mtx, dist)

        img_axes = draw_axes(img, corners2, imgpts)
        print(f"Repere 3D pour : {fname}")
        cv2_imshow(img_axes)

In [ ]:
# definition du cube
taille_cube = 3 * taille_carre_mm
axis_cube = np.float32([
    [0, 0, 0], [0, taille_cube, 0], [taille_cube, taille_cube, 0], [taille_cube, 0, 0],
    [0, 0, -taille_cube], [0, taille_cube, -taille_cube], [taille_cube, taille_cube, -taille_cube], [taille_cube, 0, -taille_cube]
])

def draw_cube(img, corners, imgpts):
    imgpts = np.int32(imgpts).reshape(-1, 2)
    img = cv2.drawContours(img, [imgpts[:4]], -1, (0, 255, 0), -3) # plancher vert
    for i, j in zip(range(4), range(4, 8)):
        img = cv2.line(img, tuple(imgpts[i]), tuple(imgpts[j]), (255, 0, 0), 3) # piliers bleus
    img = cv2.drawContours(img, [imgpts[4:]], -1, (0, 0, 255), 3) # box rouge
    return img

for fname in images_valides:
    img = cv2.imread(fname)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ret, corners = cv2.findChessboardCorners(gray, (colonnes_internes, lignes_internes), None)

    if ret:
        corners2 = cv2.cornerSubPix(gray, corners, (11,11), (-1,-1), criteria)
        ret_pnp, rvecs_img, tvecs_img = cv2.solvePnP(objp, corners2, mtx, dist)
        imgpts, jac = cv2.projectPoints(axis_cube, rvecs_img, tvecs_img, mtx, dist)

        img_cube = draw_cube(img, corners2, imgpts)
        print(f"Cube 3D pour : {fname}")
        cv2_imshow(img_cube)